# LongMemEval n=500 — engram overnight run

Phase A of the benchmark roadmap. Runs the full `longmemeval_s` split
(500 questions) against `vstash` and `engram-heuristic` baselines.

**Runtime:** ~2-3h on Colab T4 GPU (vs ~9h on Mac with MLX).

**Instructions:**
1. Runtime → Change runtime type → T4 GPU
2. Run all cells
3. Copy the final output into `RESULTS.md`

In [ ]:
# 1. Clone repos and install
!git clone https://github.com/stffns/vstash.git /content/vstash 2>/dev/null || (cd /content/vstash && git pull)
!git clone https://github.com/stffns/merken.git /content/merken 2>/dev/null || (cd /content/merken && git pull)

# Install vstash first, then engram
%pip install -e /content/vstash -q
%pip install -e /content/merken -q

# Verify GPU
import torch
print(f'CUDA available: {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'GPU: {torch.cuda.get_device_name(0)}')

In [ ]:
# 2. Verify imports work
import vstash
from engram import Memory, HeuristicWriteDecider
from experiments.retrieval.longmemeval.runner import main as lme_main
print('All imports OK')
print(f'vstash version: {vstash.__version__}')
print(f'engram version: {engram.__version__}')

In [ ]:
# 3. Sanity run (n=3, should take <2 min)
import os, subprocess, sys
os.chdir('/content/merken')

!python -m experiments.retrieval.longmemeval.runner \
    --subset longmemeval_s \
    --questions 3 \
    --seed 42 \
    --top-k 5 \
    --baseline vstash \
    --baseline engram-heuristic

In [ ]:
# 4. Record commit SHA and start time
import time, datetime

commit = !git rev-parse HEAD
branch = !git rev-parse --abbrev-ref HEAD
start_time = time.time()
start_ts = datetime.datetime.utcnow().strftime('%Y-%m-%dT%H:%M:%SZ')

print(f'Commit: {commit[0]}')
print(f'Branch: {branch[0]}')
print(f'Started: {start_ts}')

In [ ]:
# 5. FULL RUN — n=500, both baselines
# This is the real deal. ~2-3h on T4.

!python -m experiments.retrieval.longmemeval.runner \
    --subset longmemeval_s \
    --questions 500 \
    --seed 42 \
    --top-k 5 \
    --baseline vstash \
    --baseline engram-heuristic

In [ ]:
# 6. Record elapsed time
elapsed = time.time() - start_time
hours = elapsed / 3600
print(f'\n=== DONE ===')
print(f'Elapsed: {elapsed:.0f}s ({hours:.1f}h)')
print(f'Commit: {commit[0]}')
print(f'Started: {start_ts}')
print(f'\nCopy the baseline= lines above into RESULTS.md with:')
print(f'  date: {datetime.datetime.utcnow().strftime("%Y-%m-%d")}')
print(f'  commit: {commit[0][:7]}')
print(f'  n=500, seed=42, subset=longmemeval_s')
print(f'  wall-clock: {hours:.1f}h on Colab T4')